In [1]:
import numpy as np
import tensorflow as tf
import keras
from matplotlib import pyplot as plt
import altair as alt
import os
import pandas as pd
from tqdm import tqdm
import time
import faiss

from scipy.stats import normaltest
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
import Levenshtein

# import argparse
alt.data_transformers.disable_max_rows()

2026-05-05 00:02:44.554952: E external/local_xla/xla/stream_executor/plugin_registry.cc:91] Invalid plugin kind specified: FFT
2026-05-05 00:02:47.737436: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-05 00:02:49.511063: E external/local_xla/xla/stream_executor/plugin_registry.cc:91] Invalid plugin kind specified: DNN


DataTransformerRegistry.enable('default')

In [2]:
'''
Load the data and model
'''

# data_path = "../data/test-species/tensorflow_datasets/vagococcus-fluvialis_dataset"
data_path = "../data/test-species/tensorflow_datasets/streptococcus-pneumoniae_dataset"
# data_path = "/scratch/project_465002309/rosstheo/genome_transformer_dev/data/gene_sequences/test_dataset"
model_path = "models/geneAE_curious-totem-5139_fold0.keras"


gene_ae = tf.keras.models.load_model(model_path)
gene_dataset = tf.data.Dataset.load(data_path)
gene_dataset = gene_dataset.filter(lambda g,d,c: tf.strings.length(g) <= 5000)
_,gene_dataset = gene_ae.preprocess_dataset(gene_dataset, validation_data=gene_dataset, batch_size=64, weighted=False, shuffle=False)



2026-05-05 00:06:02.757286: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 63906 MB memory:  -> device: 0, name: AMD Instinct MI250X, pci bus id: 0000:d6:00.0
2026-05-05 00:06:52.636323: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
'''
Collect label data
'''
## Initialize the lists to store data
cog_labels = []
domain_labels = []
reconstruction_targets = []

## Populate the lists
for g,labels in tqdm(gene_dataset):
    # val_genes.append(g)
    
    _domain,_cat,_recon_target = labels
    cog_labels.append(_cat)
    domain_labels.append(_domain)
    reconstruction_targets.append(_recon_target)

# ## Convert them to arrays
# val_genes = np.concatenate(val_genes, axis=0)
cog_labels = np.concatenate(cog_labels, axis=0)
domain_labels = np.concatenate(domain_labels, axis=0)
reconstruction_targets = np.concatenate(reconstruction_targets, axis=0)


1529it [00:22, 68.44it/s]2026-05-05 00:07:15.506169: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
1531it [00:22, 66.96it/s]


In [4]:
'''
Compute predictions
'''
embeddings, cog_preds, reconstructions = gene_ae.predict(gene_dataset)

1531/1531 ━━━━━━━━━━━━━━━━━━━━ 407s 262ms/step


/users/rosstheo/.local/lib/python3.10/site-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [5]:
'''
Evaluate COG predictions
'''


def eval_cog_preds(preds, labels):
    '''
    Evaluate the COG category predictions of the gene autoencoder model.
    
    Inputs:
        model (GeneTransformer) : keras model for the gene transformer model.
        dataset (tensorflow.data.Dataset) : tensorflow dataset containing the genes, COG categories, and protein domains.
    '''

    ## Define prediction and label arrays
    preds = (preds > 0.5).astype(int)
    correct_mask = np.all(preds == labels, axis=1)
    print(f"genes completely correct:\t\t {correct_mask.sum():5d}\t ({correct_mask.mean()*100:3.2f}%)")

    
    ## Isolate genes that had functional labels
    labeled_mask = labels.sum(axis=1) > 0
    labeled_preds = preds[labeled_mask]
    labeled_labels = labels[labeled_mask].astype(int)

    full_correct = np.all(labeled_preds == labeled_labels, axis=1)
    not_full_corr_mask = ~np.all(labeled_preds == labeled_labels, axis=1)
    labelled_no_pred = labeled_preds.sum(axis=1) == 0
    
    # print(f"labeled genes completely correct:\t {full_correct.sum():5d}\t ({full_correct.mean()*100:3.2f}%)")
    # # print("percent no pred:", (val_cat_preds.sum(axis=1)==0).mean())
    
    
    # print(f"genes with labels:\t\t\t", 
    #       f"{(labels.sum(axis=1)!=0).sum():5d}\t",
    #       f"({(labels.sum(axis=1)!=0).mean()*100:3.2f}%)")
    
    # print(f"genes with no prediction:\t\t ",
    #       f"{(preds.sum(axis=1)==0).sum():4d}\t ",
    #       f"({(preds.sum(axis=1)==0).mean()*100:3.2f}%)")
    
    # print(f"genes with labels and no prediction:\t ", 
    #       f"{labelled_no_pred.sum():4d}\t ",
    #       f"({labelled_no_pred.mean()*100:3.2f}%)")

    ## Create a csv for saving summary information
    csv_str = "\ttotal\tpercent\n"
    csv_str += f"total genes\t{len(preds)}\t\n"

    # Genes completely correct
    csv_str += "genes with labels\t"
    csv_str += f"{(labels.sum(axis=1)!=0).sum()}\t"
    csv_str += f"{(labels.sum(axis=1)!=0).mean()*100}%\n"

    csv_str += "genes with no prediction\t"
    csv_str += f"{(preds.sum(axis=1)==0).sum()}\t"
    csv_str += f"{(preds.sum(axis=1)==0).mean()*100}%"

    csv_str += "genes with labels and no prediction\t"
    csv_str += f"{labelled_no_pred.sum()}\t"
    csv_str += f"{labelled_no_pred.mean()*100}"

    ## Find all unique combination of COG category
    categs = np.asarray([c for c in 'ABCDEFGHIJKLMNOPQTUVWYZ'])
    categ_df = pd.DataFrame(index=categs, columns=["Count", "F1", "Precision", "Recall", "Accuracy"])
    categ_df.index.name = "Category"

    for c in range(labels.shape[1]):
        _true = labeled_labels[:,c]
        cat_letter = categs[c]

        if _true.sum() != 0:
            # _pred = val_cat_preds[:,c]
            _pred = labeled_preds[:,c]
    
            categ_df.loc[cat_letter,"Count"] = _true.sum()
            categ_df.loc[cat_letter,"F1"] = metrics.f1_score(_true,_pred)
            categ_df.loc[cat_letter,"Precision"] = metrics.precision_score(_true,_pred)
            categ_df.loc[cat_letter,"Recall"] = metrics.recall_score(_true,_pred)
            categ_df.loc[cat_letter,"Accuracy"] = metrics.accuracy_score(_true,_pred)   

    return csv_str, categ_df

eval_cog_preds(cog_preds, cog_labels)



genes completely correct:		 21224	 (21.66%)


/users/rosstheo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/users/rosstheo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/users/rosstheo/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


('\ttotal\tpercent\ntotal genes\t97984\t\ngenes with labels\t71010\t72.47101567602874%\ngenes with no prediction\t45704\t46.64435009797518%genes with labels and no prediction\t32178\t45.31474440219687',
          Count        F1 Precision    Recall  Accuracy
 Category                                              
 A          280       0.0       0.0       0.0  0.996057
 B          NaN       NaN       NaN       NaN       NaN
 C         2974  0.078017  0.122832  0.057162  0.943416
 D         1699  0.017363  0.111111  0.009417  0.974497
 E         5504  0.156477   0.17045  0.144622  0.879144
 F         6743  0.128312  0.322581  0.080083  0.896677
 G         8129  0.260131  0.233408  0.293763  0.808703
 H         2782  0.067149  0.119266  0.046729  0.949134
 I         1290  0.003927  0.012605  0.002326  0.978566
 J         6542  0.146367  0.521048  0.085142  0.908506
 K         6431  0.271968  0.313082  0.240398  0.883439
 L         8862  0.173684  0.276491  0.126608  0.849655
 M         62

In [ ]:
'''
Evaluate reconstructions
'''

def eval_reconstructions(preds, labels):
    '''
    Evaluate the gene reconstructions.
    '''
    ## Define the vocabulary and the reconstruction indices
    pred_idx = np.argmax(preds, axis=2)

    ## Compute the gene lengths and the reconstruction accuracies per gene
    true_tokens = pred_idx == labels
    mask = labels != 0
    true_tokens = true_tokens * mask       # Mask the token predictions
    gene_lens = mask.sum(axis=1, keepdims=True)
    print("DEBUG:", true_tokens.shape, gene_lens.shape)
    gene_accs = true_tokens.sum(axis=1) / gene_lens

    # fig,ax = plt.subplots(1,1)
    # ax.hist(gene_accs, bins=50)
    # ax.set_xlabel("Recon. Acc.")
    # # fig.savefig("temp.png")

    max_ind = np.argmax(gene_accs) - 1
    # print("debug:")
    # print(preds[max_ind,...])
    # print(labels[max_ind,...])
    # print(true_tokens[max_ind,...])
    # print(gene_lens[max_ind,...])

    ## Compute the GC content of each gene
    c_mask = labels == 4
    g_mask = labels == 5
    gc_mask = c_mask | g_mask
    gc_frac = gc_mask.sum(axis=1) / mask.sum(axis=1)

    ## Create a histogram of the reconstruction accuracies
    df = pd.DataFrame({"recon_acc":gene_accs,
                       "gene_lens":gene_lens,
                       "gc_frac":gc_frac})
    base = alt.Chart(df)
    bar_hist = base.mark_bar().encode(
        x = alt.X("recon_acc:Q", title="Reconstruction Accuracy", bin=alt.Bin(maxbins=30)),
        y = alt.Y("count()")
    )
    len_plot = base.mark_circle().encode(
        x = alt.X("gene_lens:Q", title="Gene Length (nt)"),
        y = alt.Y("recon_acc:Q", title="Reconstruction Accuracy")
    )
    gc_plot = base.mark_circle().encode(
        x = alt.X("gc_frac:Q", title="GC Content"),
        y = alt.Y("recon_acc:Q", title="Reconstruction Accuracy")
    )  
    out_plot = bar_hist | len_plot | gc_plot
    
    return df, out_plot

_, recon_chart = eval_reconstructions(reconstructions, reconstruction_targets)
recon_chart


DEBUG: (97984, 5000) (97984, 1)


In [ ]:
'''
Evaluate the domain-based clustering
'''

def eval_domains(z, y):
    '''
    Evaluate the domains 
    '''

    ## Initialize FAISS
    faiss_index = faiss.IndexFlatL2(2048)
    faiss_index.add(z)

    ## Loop through each domain to compute centers
    domain_centers = np.zeros((y.shape[1],2048))
    domain_counts = []
    nearest_neighbor_matching_fracs = []
    
    # for dx in tqdm(range(y.shape[1])[-50:]):
    for dx in tqdm(range(y.shape[1])):

        ## Find the genes containing this domain
        domain_mask = y[:,dx].astype(bool)
        n_domains = int( domain_mask.sum() )

        if n_domains != 0:
            ## Find the mean/center of this domain's embeddings
            domain_mean = z[domain_mask,:].mean(axis=0)[None,:]

            ## Find the n_domains closest embeddings to the center
            D,I = faiss_index.search(domain_mean, n_domains)

            ## Determine if the n_domains nearest neighbors contain the query domain
            label_match_flags = y[np.squeeze(I), dx]
            nearest_neighbor_matching_fracs.append(np.mean(label_match_flags))
            domain_counts.append(n_domains)

    ## Make an output plot
    # Define the dataframe to plot
    df = pd.DataFrame({'count':domain_counts, 
                       'frac':nearest_neighbor_matching_fracs})

    # Define the chart bases and scales
    base = alt.Chart(df)
    bar_base = base.mark_bar(binSpacing=0)
    
    frac_scale = alt.Scale(domain=(0,1))
    count_scale = alt.Scale(domain=(0,max(domain_counts)*1.05))
    
    # Create the scatterplot
    points = base.mark_circle().encode(
        x = alt.X('count:Q', title="Total domain instances").scale(count_scale, type="log"),
        y = alt.Y('frac:Q', title="Fraction included in nearest neighbors").scale(frac_scale)
    )

    # Create a hitogram of the counts
    top_bar = bar_base.encode(
        x = alt.X('count:Q').scale(count_scale, type="log").bin(
            maxbins=30, extent=count_scale.domain
        ).title(""),
        y = alt.Y('count():Q').stack(None).title("")
    ).properties(height=60)

    # Create a histogram of the fractions
    right_bar = bar_base.encode(
        y = alt.Y('frac:Q').scale(frac_scale).bin(
            maxbins=30, extent=frac_scale.domain
        ).title(""),
        x = alt.X('count():Q').stack(None).title("")
    ).properties(width=60)

    # Format the final chart
    chart = top_bar & (points | right_bar)

    return nearest_neighbor_matching_fracs, chart


nn_matching, domain_chart = eval_domains(embeddings, domain_labels)
domain_chart
